In [1]:
# 1. celica - dodamo vse potrebne knjižnice - PyTorch za nevronske mreže, OpenCV za delo s slikami in Matplotlib za risanje grafov

import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# preverimo ali ima naš rač grafično kartico (GPU) za hitrejše učenje, drugače uporabimo procesor (CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Učeneje bo potekalo na napravi: {device}")

Učeneje bo potekalo na napravi: cpu


In [ ]:
# Namestimo split-folders neposredno iz beležnice, če je še ni
!pip install split-folders

import splitfolders

# Nastavi ime tvoje mape, kjer imaš trenutno shranjene surove slike obrazov
input_folder = "dataset\surovi_podatki"  
output_folder = "podatki_za_model"

# Razdelitev podatkov z določenim naključnim semenom (seed) za ponovljivost rezultatov
splitfolders.ratio(input_folder, output=output_folder, seed=1337, ratio=(.7, .15, .15))
print("Slike so uspešno razdeljene v mape train, val in test!")

Slike so uspešno razdeljene v mape train, val in test!


In [7]:
# Definiramo transformacije za učno množico 
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5), # argumentacija: zrcaljenje
    transforms.RandomRotation(15), # rotacija na max 15 stopinj
    transforms.ToTensor(), # pretvorba v tenzor (normalizira pixle v [0.0, 1.0])
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]) # normalizacija za RestNet
])

val_transforms = transforms.Compose([
    transforms.Resize((224, 224)), 
    transforms.ToTensor(), 
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# nalozimo podatke iz map
train_dataset = datasets.ImageFolder(root='podatki_za_model/train', transform=train_transforms)
val_dataset = datasets.ImageFolder(root='podatki_za_model/val', transform=val_test_transforms)
test_dataset = datasets.ImageFolder(root='podatki_za_model/test', transform=val_test_transforms)

BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

imena_oseb = train_dataset.classes
print(f"Zaznane osebe za prepoznavo: {imena_oseb}")
print(f"Število slik - Učna: {len(train_dataset)}, Validacijska: {len(val_dataset)}, Testna: {len(test_dataset)}")


Zaznane osebe za prepoznavo: ['iris', 'manja']
Število slik - Učna: 25, Validacijska: 4, Testna: 7


In [8]:
# nalozimo v naprej naučen model ResNet18
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

for param in model.parameters(): 
    param.requires_grad = False

stevilo_vhodnih_znacilnosti = model.fc.in_features
stevilo_razredov = len(imena_oseb)

model.fc = nn.Linear(stevilo_vhodnih_znacilnosti, stevilo_razredov)

model = model.to(device)
print("Model ResNet18 je uspešno naložen in prilagojen podatkom")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\Uporabnik/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100.0%


Model ResNet18 je uspešno naložen in prilagojen podatkom
